# Explore here

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
out_path = "../data/raw/playstore_reviews.csv"

df = pd.read_csv(url)
df.to_csv(out_path, index=False)

print(f"Saved {len(df)} rows to {out_path}")
df.head()

Saved 891 rows to ../data/raw/playstore_reviews.csv


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [3]:
df = df.drop(columns=["package_name"])


In [7]:

print(df.columns)

Index(['review', 'polarity'], dtype='str')


In [8]:
df["column"] = df["review"].str.strip().str.lower()

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["polarity"])  
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test, 80% train
    random_state=42,    # reproducible split
    stratify=y          # keeps class balance (recommended for classification)
)

In [11]:
from sklearn.feature_extraction.text import CountVectorizer



In [12]:
vec_model = CountVectorizer(stop_words="english")
X_train_vec = vec_model.fit_transform(X_train["review"])
X_test_vec = vec_model.transform(X_test["review"])

print(X_train_vec.shape, X_test_vec.shape)

(712, 3272) (179, 3272)


In [15]:
print(X_train_vec[:5].toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [16]:
print(vec_model.get_feature_names_out()[:20])   # first 20 words

['04' '0x' '10' '100' '101' '11' '113mb' '1186' '12' '125' '13' '14' '15'
 '15mb' '16' '17' '180k' '1990s' '1day' '1lac']


In [ ]:
row0 = X_train_vec[0]
idx = row0.nonzero()[1]
words = vec_model.get_feature_names_out()[idx]
counts = row0.data
print(list(zip(words, counts))[:20])row0 = X_train_vec[0]
idx = row0.nonzero()[1]
words = vec_model.get_feature_names_out()[idx]
counts = row0.data
print(list(zip(words, counts))[:20])

[('best', np.int64(1)), ('app', np.int64(1)), ('believe', np.int64(1)), ('free', np.int64(1)), ('treat', np.int64(1)), ('paid', np.int64(1))]


In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



In [14]:
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

y_pred = nb_model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8547486033519553

Classification report:
               precision    recall  f1-score   support

           0       0.84      0.96      0.90       117
           1       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179


Confusion matrix:
 [[112   5]
 [ 21  41]]


In [18]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

In [19]:


param_grid = {"alpha": np.logspace(-3, 1, 20)}  # 0.001 to 10
grid = GridSearchCV(
    MultinomialNB(),
    param_grid=param_grid,
    cv=5,
    scoring="f1",   # for binary classes 0/1
    n_jobs=-1
)

grid.fit(X_train_vec, y_train)

print("Best alpha:", grid.best_params_["alpha"])
print("Best CV F1:", grid.best_score_)

best_nb = grid.best_estimator_
y_pred = best_nb.predict(X_test_vec)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nTest report:\n", classification_report(y_test, y_pred))

Best alpha: 0.029763514416313176
Best CV F1: 0.7090709970417681
Test Accuracy: 0.8379888268156425

Test report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88       117
           1       0.84      0.66      0.74        62

    accuracy                           0.84       179
   macro avg       0.84      0.80      0.81       179
weighted avg       0.84      0.84      0.83       179



In [20]:
from pathlib import Path
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Final evaluation with the tuned model from GridSearchCV
final_model = grid.best_estimator_
y_pred_final = final_model.predict(X_test_vec)

print("Final tuned alpha:", final_model.alpha)
print("Final test accuracy:", accuracy_score(y_test, y_pred_final))
print("\nFinal classification report:\n", classification_report(y_test, y_pred_final))
print("\nFinal confusion matrix:\n", confusion_matrix(y_test, y_pred_final))

# Show some mistakes for error analysis
errors = X_test.copy()
errors["true_label"] = y_test.values
errors["pred_label"] = y_pred_final
errors = errors[errors["true_label"] != errors["pred_label"]]

print("\nMisclassified examples (up to 10):")
print(errors[["review", "true_label", "pred_label"]].head(10))

# Save model artifacts
Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(vec_model, "../models/count_vectorizer.joblib")
joblib.dump(final_model, "../models/multinomial_nb_tuned.joblib")
print("\nSaved ../models/count_vectorizer.joblib and ../models/multinomial_nb_tuned.joblib")

Final tuned alpha: 0.029763514416313176
Final test accuracy: 0.8379888268156425

Final classification report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88       117
           1       0.84      0.66      0.74        62

    accuracy                           0.84       179
   macro avg       0.84      0.80      0.81       179
weighted avg       0.84      0.84      0.83       179


Final confusion matrix:
 [[109   8]
 [ 21  41]]

Misclassified examples (up to 10):
                                                review  true_label  pred_label
147   suggestion. i given 5 stars to this game,  be...           1           0
836                usefull no others app like this....           1           0
634   nice update really nice update this update is...           1           0
659   it's all awesome. can you devs please add mor...           1           0
453   very nice app best feature is all friends get...           1           0
481

## Final Metrics and Model Choice

This section summarizes the final model used for submission.

- Final model: Tuned `MultinomialNB` (`grid.best_estimator_`)
- Tuned hyperparameter: `alpha = grid.best_params_["alpha"]`
- Reported final test metrics: Accuracy, Classification Report, and Confusion Matrix (from the previous cell)

Why this model was chosen:
- Text features are count-based (`CountVectorizer`), which matches `MultinomialNB` assumptions.
- `alpha` was tuned with 5-fold cross-validation to improve generalization rather than selecting by one split only.
- The tuned model is selected because it optimizes the target validation metric used in tuning (`f1`).

In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

summary_pred = final_model.predict(X_test_vec)

print("=== FINAL SUBMISSION SUMMARY ===")
print(f"Model: MultinomialNB (tuned)")
print(f"Chosen alpha: {final_model.alpha}")
print(f"Accuracy: {accuracy_score(y_test, summary_pred):.4f}")
print("\nClassification report:\n", classification_report(y_test, summary_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, summary_pred))

=== FINAL SUBMISSION SUMMARY ===
Model: MultinomialNB (tuned)
Chosen alpha: 0.029763514416313176
Accuracy: 0.8380

Classification report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88       117
           1       0.84      0.66      0.74        62

    accuracy                           0.84       179
   macro avg       0.84      0.80      0.81       179
weighted avg       0.84      0.84      0.83       179


Confusion matrix:
 [[109   8]
 [ 21  41]]
